# **SETUP**

In [1]:
import pathlib
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

PROJECT_ROOT = pathlib.Path().absolute().parent

train = pd.read_parquet(PROJECT_ROOT / "data" / "train.parquet")
ohe_train = pd.read_parquet(PROJECT_ROOT / "data" / "features" / "002-one-hot-categoricals" / "train.parquet")
std_train = pd.read_parquet(PROJECT_ROOT / "data" / "features" / "003-standard-scale-numerics" / "train.parquet")
kbins_train = pd.read_parquet(PROJECT_ROOT / "data" / "features" / "005-kbins-discretize-numerics" / "train.parquet")
ohd_train = pd.read_parquet(PROJECT_ROOT / "data" / "features" / "006-one-hot-driver-high-cardinality" / "train.parquet")
st_train = pd.read_parquet(PROJECT_ROOT / "data" / "features" / "007-spline-transform-numerics" / "train.parquet")

test = pd.read_parquet(PROJECT_ROOT / "data" / "test.parquet")
ohe_test = pd.read_parquet(PROJECT_ROOT / "data" / "features" / "002-one-hot-categoricals" / "test.parquet")
std_test = pd.read_parquet(PROJECT_ROOT / "data" / "features" / "003-standard-scale-numerics" / "test.parquet")
kbins_test = pd.read_parquet(PROJECT_ROOT / "data" / "features" / "005-kbins-discretize-numerics" / "test.parquet")
ohd_test = pd.read_parquet(PROJECT_ROOT / "data" / "features" / "006-one-hot-driver-high-cardinality" / "test.parquet")
st_test = pd.read_parquet(PROJECT_ROOT / "data" / "features" / "007-spline-transform-numerics" / "test.parquet")

X_train = (
    train
    .reset_index()[["id"]]
    .merge(ohe_train, how="left", on="id")
    .merge(std_train, how="left", on="id")
    .merge(kbins_train, how="left", on="id")
    .merge(ohd_train, how="left", on="id")
    .merge(st_train, how="left", on="id")
    .drop(columns=["id"])
)
X_test = (
    test
    .reset_index()[["id"]]
    .merge(ohe_test, how="left", on="id")
    .merge(std_test, how="left", on="id")
    .merge(kbins_test, how="left", on="id")
    .merge(ohd_test, how="left", on="id")
    .merge(st_test, how="left", on="id")
    .drop(columns="id")
)
y_train = train["PitNextLap"]

cv = pd.read_parquet(PROJECT_ROOT / "data" / "cv.parquet")

logreg = LogisticRegression(max_iter=5000, solver="lbfgs", C=0.1, random_state=123)

# **OOF PREDICTIONS**

In [2]:
scores = []
oof = pd.Series(index=train.index, dtype=float, name="oof")

for k in sorted(cv.outer_fold.unique()):
    is_val = cv["outer_fold"] == k
    logreg.fit(X_train[~is_val], y_train[~is_val])

    oof[is_val] = logreg.predict_proba(X_train[is_val])[:, 1]
    scores.append(roc_auc_score(y_train[is_val], oof[is_val]))

print(scores)
print("OOF ROC-AUC:", roc_auc_score(y_train, oof))

[0.9200153549230151, 0.9200407547379261, 0.9186812147681278, 0.9181728476209223, 0.9187828531365756]
OOF ROC-AUC: 0.919142033963815


# **TEST PREDICTIONS**

In [3]:
logreg.fit(X_train, y_train)
preds = pd.Series(logreg.predict_proba(X_test)[:, 1], index=test.index, dtype=float, name="preds")

# **EXPORT**

In [4]:
model_name = "004-linear-fe"
(PROJECT_ROOT / "data" / "predictions" / model_name).mkdir(exist_ok=True)

oof.to_frame().to_parquet(PROJECT_ROOT / "data" / "predictions" / model_name / "train.parquet")
preds.to_frame().to_parquet(PROJECT_ROOT / "data" / "predictions" / model_name / "test.parquet")